# Leaf Dieback Detection Model v4

## Improvements over v3:
- MobileNetV2 backbone (simpler, better generalization)
- Stronger class weights for healthy class (2x boost)
- Label smoothing to prevent overconfidence
- Focal loss with optimized gamma
- Learning rate warmup schedule

## Dataset:
- Train: ~2850 images per class (balanced with augmentation)
- Val/Test: Original images only (no augmentation to avoid data leaking)
- Classes: healthy, leaf_die_back, not_cocount

## 1. Setup and Imports

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Configuration

In [ ]:
# Configuration
MODEL_VERSION = "v4"
MODEL_NAME = "leaf_dieback"
BASE_DIR = r"C:\Users\Tharindu Nandun\Desktop\Research\Research\ml"
DATA_DIR = os.path.join(BASE_DIR, "data", "processed", "leaf_dieback_v1")
MODEL_DIR = os.path.join(BASE_DIR, "models", f"{MODEL_NAME}_{MODEL_VERSION}")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_PHASE1 = 20  # Frozen base
EPOCHS_PHASE2 = 30  # Fine-tuning

os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Model directory: {MODEL_DIR}")

## 3. Dataset Analysis

In [ ]:
# Analyze dataset distribution
print("Dataset Distribution:")
print("="*50)

splits_data = {}
for split in ['train', 'val', 'test']:
    split_path = os.path.join(DATA_DIR, split)
    splits_data[split] = {}
    print(f"\n{split.upper()}:")
    for cls in sorted(os.listdir(split_path)):
        cls_path = os.path.join(split_path, cls)
        count = len(os.listdir(cls_path))
        splits_data[split][cls] = count
        print(f"  {cls}: {count}")

# Visualize distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for idx, split in enumerate(['train', 'val', 'test']):
    classes = list(splits_data[split].keys())
    counts = list(splits_data[split].values())
    colors = ['#2ecc71', '#e74c3c', '#3498db']
    axes[idx].bar(classes, counts, color=colors)
    axes[idx].set_title(f'{split.upper()} Distribution')
    axes[idx].set_ylabel('Count')
    for i, v in enumerate(counts):
        axes[idx].text(i, v + 10, str(v), ha='center')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'dataset_distribution.png'), dpi=150)
plt.show()

## 4. Data Generators

In [ ]:
# Data generators with augmentation for training only
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen = val_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

class_names = list(train_gen.class_indices.keys())
num_classes = len(class_names)
print(f"\nClasses: {class_names}")
print(f"Train samples: {train_gen.samples}")
print(f"Val samples: {val_gen.samples}")
print(f"Test samples: {test_gen.samples}")

## 5. Sample Images

In [ ]:
# Show sample images from each class
fig, axes = plt.subplots(3, 4, figsize=(12, 9))

for row, cls in enumerate(class_names):
    cls_path = os.path.join(DATA_DIR, 'train', cls)
    images = os.listdir(cls_path)[:4]
    for col, img_name in enumerate(images):
        img_path = os.path.join(cls_path, img_name)
        img = plt.imread(img_path)
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cls, fontsize=12, fontweight='bold')

plt.suptitle('Sample Images from Each Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'sample_images.png'), dpi=150)
plt.show()

## 6. Class Weights Calculation

In [ ]:
# Calculate class weights - boost healthy significantly
train_labels = train_gen.classes
class_weights_array = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = {i: w for i, w in enumerate(class_weights_array)}

# Extra boost for healthy class
healthy_idx = list(train_gen.class_indices.keys()).index('healthy')
class_weights[healthy_idx] *= 2.0  # Double the weight for healthy

print("Class weights (with healthy boost):")
for cls, idx in train_gen.class_indices.items():
    print(f"  {cls}: {class_weights[idx]:.4f}")

## 7. Loss Functions

In [ ]:
# Focal Loss for handling class imbalance
def focal_loss(gamma=2.0, alpha=0.25):
    def focal_loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        cross_entropy = -y_true * tf.math.log(y_pred)
        focal_weight = alpha * tf.pow(1 - y_pred, gamma) * y_true
        focal_loss = tf.reduce_sum(focal_weight * cross_entropy, axis=-1)
        return focal_loss
    return focal_loss_fn

# Label smoothing cross entropy
def label_smoothing_loss(smoothing=0.1):
    def loss_fn(y_true, y_pred):
        num_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_true_smooth = y_true * (1.0 - smoothing) + smoothing / num_classes
        return tf.keras.losses.categorical_crossentropy(y_true_smooth, y_pred)
    return loss_fn

print("Loss functions defined:")
print("  - Focal Loss (gamma=2.5, alpha=0.3) for Phase 2")
print("  - Label Smoothing Loss (smoothing=0.1) for Phase 1")

## 8. Build Model

In [ ]:
# Learning rate warmup schedule
def lr_schedule(epoch, lr):
    warmup_epochs = 5
    if epoch < warmup_epochs:
        return 1e-4 * (epoch + 1) / warmup_epochs
    else:
        return lr

# Build model with MobileNetV2
def build_model():
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(*IMG_SIZE, 3)
    )

    # Freeze base model
    base_model.trainable = False

    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model, base_model

print("Building model...")
model, base_model = build_model()
model.summary()

## 9. Phase 1: Training with Frozen Base

In [ ]:
print("="*60)
print("PHASE 1: Training with frozen base layers")
print("="*60)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=label_smoothing_loss(smoothing=0.1),
    metrics=['accuracy']
)

callbacks_phase1 = [
    ModelCheckpoint(
        os.path.join(MODEL_DIR, 'phase1_model.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    LearningRateScheduler(lr_schedule, verbose=0)
]

history1 = model.fit(
    train_gen,
    epochs=EPOCHS_PHASE1,
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=callbacks_phase1,
    verbose=1
)

## 10. Phase 2: Fine-tuning

In [ ]:
print("="*60)
print("PHASE 2: Fine-tuning top layers")
print("="*60)

# Unfreeze last 30 layers of base model
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

print(f"Total layers: {len(base_model.layers)}")
print(f"Trainable layers: {sum(1 for l in base_model.layers if l.trainable)}")

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=focal_loss(gamma=2.5, alpha=0.3),
    metrics=['accuracy']
)

callbacks_phase2 = [
    ModelCheckpoint(
        os.path.join(MODEL_DIR, 'best_model.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1
    )
]

history2 = model.fit(
    train_gen,
    epochs=EPOCHS_PHASE2,
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=callbacks_phase2,
    verbose=1
)

## 11. Training Curves

In [ ]:
# Combine histories
history = {
    'accuracy': history1.history['accuracy'] + history2.history['accuracy'],
    'val_accuracy': history1.history['val_accuracy'] + history2.history['val_accuracy'],
    'loss': history1.history['loss'] + history2.history['loss'],
    'val_loss': history1.history['val_loss'] + history2.history['val_loss']
}

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history['accuracy'], label='Train Accuracy', linewidth=2, color='#3498db')
axes[0].plot(history['val_accuracy'], label='Val Accuracy', linewidth=2, color='#e74c3c')
axes[0].axvline(x=len(history1.history['accuracy'])-1, color='gray', linestyle='--', label='Fine-tuning Start')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history['loss'], label='Train Loss', linewidth=2, color='#3498db')
axes[1].plot(history['val_loss'], label='Val Loss', linewidth=2, color='#e74c3c')
axes[1].axvline(x=len(history1.history['loss'])-1, color='gray', linestyle='--', label='Fine-tuning Start')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'training_curves.png'), dpi=150)
plt.show()

print(f"\nBest Val Accuracy: {max(history['val_accuracy'])*100:.2f}%")

## 12. Load Best Model & Evaluate

In [ ]:
print("="*60)
print("EVALUATION ON TEST SET")
print("="*60)

# Load best model
best_model_path = os.path.join(MODEL_DIR, 'best_model.keras')
if os.path.exists(best_model_path):
    model = keras.models.load_model(best_model_path, compile=False)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    print(f"Loaded best model from: {best_model_path}")

# Evaluate
test_gen.reset()
test_loss, test_acc = model.evaluate(test_gen, verbose=0)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

## 13. Classification Report

In [ ]:
# Get predictions
test_gen.reset()
y_pred_probs = model.predict(test_gen, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes

# Classification report
print("="*60)
print("CLASSIFICATION REPORT")
print("="*60)
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
print(classification_report(y_true, y_pred, target_names=class_names))

## 14. Confusion Matrix

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            annot_kws={'size': 14})
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

print("\nConfusion Matrix:")
print(cm)

## 15. Per-Class Metrics Visualization

In [ ]:
# Per-class metrics bar chart
metrics_data = {
    'Precision': [report[cls]['precision'] for cls in class_names],
    'Recall': [report[cls]['recall'] for cls in class_names],
    'F1-Score': [report[cls]['f1-score'] for cls in class_names]
}

x = np.arange(len(class_names))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width, metrics_data['Precision'], width, label='Precision', color='#3498db')
bars2 = ax.bar(x, metrics_data['Recall'], width, label='Recall', color='#2ecc71')
bars3 = ax.bar(x + width, metrics_data['F1-Score'], width, label='F1-Score', color='#e74c3c')

ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Per-Class Metrics (Precision, Recall, F1-Score)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'per_class_metrics.png'), dpi=150)
plt.show()

## 16. Supervisor Requirements Check

In [ ]:
print("="*60)
print("SUPERVISOR REQUIREMENTS CHECK")
print("="*60)

all_good = True
results = []

for cls in class_names:
    p = report[cls]['precision']
    r = report[cls]['recall']
    f1 = report[cls]['f1-score']
    diff = max(p, r, f1) - min(p, r, f1)
    
    status = "PASS" if diff < 0.15 else "FAIL"
    if diff >= 0.15:
        all_good = False
    
    results.append({
        'class': cls,
        'precision': p,
        'recall': r,
        'f1': f1,
        'diff': diff,
        'status': status
    })
    
    print(f"{cls}:")
    print(f"  Precision: {p:.4f}")
    print(f"  Recall:    {r:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  P-R-F1 Diff: {diff:.4f} [{status}]")
    print()

macro_f1 = report['macro avg']['f1-score']
acc_f1_diff = abs(test_acc - macro_f1)
acc_status = "PASS" if acc_f1_diff < 0.1 else "FAIL"

print(f"Overall Metrics:")
print(f"  Test Accuracy: {test_acc:.4f}")
print(f"  Macro F1:      {macro_f1:.4f}")
print(f"  Acc-F1 Diff:   {acc_f1_diff:.4f} [{acc_status}]")

print("\n" + "="*60)
if all_good and acc_f1_diff < 0.1:
    print("ALL REQUIREMENTS MET!")
else:
    print("Some requirements not met - may need further tuning")
print("="*60)

## 17. Save Results

In [ ]:
# Save class metrics
with open(os.path.join(MODEL_DIR, 'class_metrics.csv'), 'w') as f:
    f.write("Class,Precision,Recall,F1-Score,Support,P-R-F1 Diff,Status\n")
    for r in results:
        sup = report[r['class']]['support']
        f.write(f"{r['class']},{r['precision']:.4f},{r['recall']:.4f},{r['f1']:.4f},{sup},{r['diff']:.4f},{r['status']}\n")

# Save model info
model_info = {
    "model_name": f"{MODEL_NAME}_{MODEL_VERSION}",
    "base_model": "MobileNetV2",
    "input_size": list(IMG_SIZE) + [3],
    "classes": class_names,
    "test_accuracy": float(test_acc),
    "test_loss": float(test_loss),
    "macro_f1": float(macro_f1),
    "per_class_metrics": {
        cls: {
            "precision": float(report[cls]['precision']),
            "recall": float(report[cls]['recall']),
            "f1-score": float(report[cls]['f1-score'])
        } for cls in class_names
    },
    "supervisor_requirements": {
        "p_r_f1_close": all_good,
        "acc_f1_close": acc_f1_diff < 0.1
    }
}

with open(os.path.join(MODEL_DIR, 'model_info.json'), 'w') as f:
    json.dump(model_info, f, indent=2)

print(f"Results saved to: {MODEL_DIR}")
print("\nFiles saved:")
for f in os.listdir(MODEL_DIR):
    print(f"  - {f}")

## 18. Summary

In [ ]:
print("="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"\nModel: {MODEL_NAME}_{MODEL_VERSION}")
print(f"Base Model: MobileNetV2")
print(f"\nDataset:")
print(f"  Train: {train_gen.samples} images")
print(f"  Val: {val_gen.samples} images")
print(f"  Test: {test_gen.samples} images")
print(f"\nResults:")
print(f"  Test Accuracy: {test_acc*100:.2f}%")
print(f"  Macro F1: {macro_f1*100:.2f}%")
print(f"\nPer-Class Performance:")
for cls in class_names:
    p = report[cls]['precision']
    r = report[cls]['recall']
    f1 = report[cls]['f1-score']
    print(f"  {cls}: P={p:.2f}, R={r:.2f}, F1={f1:.2f}")
print(f"\nModel saved to: {MODEL_DIR}")
print("="*60)